In [ ]:
from transformers import AutoTokenizer as _AutoTokenizer
import os as _os
from tokenizers import Tokenizer as _HFTokenizer
from pathlib import Path

_hf_cache = Path.home() / ".cache" / "huggingface"
_snapshot = (
    _hf_cache / "hub"
    / "models--nvidia--Llama-3_3-Nemotron-Super-49B-v1_5"
    / "snapshots"
    / "420ba7d28211abf116b8b103ab700d92619daf98"
)

def _load_local_nemotron_tokenizer():
    for _var in ("HF_HOME", "HF_HUB_CACHE", "TRANSFORMERS_CACHE", "HF_DATASETS_CACHE"):
        _os.environ.pop(_var, None)
    _os.environ["HF_HOME"] = str(_hf_cache)

    try:
        from transformers import PreTrainedTokenizerFast
        return PreTrainedTokenizerFast.from_pretrained(str(_snapshot))
    except Exception:
        tokenizer_json = _snapshot / "tokenizer.json"
        if not tokenizer_json.exists():
            raise
        return _HFTokenizer.from_file(str(tokenizer_json))


_nemotron_tokenizer = _load_local_nemotron_tokenizer()

def _count_nemotron_tokens(text: str) -> int:
    return len(_nemotron_tokenizer.encode(text))

In [ ]:


def find_repo_root(start=None):
    roots_to_try = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if start is not None:
        roots_to_try.insert(0, Path(start))

    seen = set()
    for base in roots_to_try:
        current = base.resolve()
        while True:
            if current in seen:
                break
            seen.add(current)
            if (current / 'outputs' / 'runs').exists():
                return current
            if current.parent == current:
                break
            current = current.parent

    return Path.cwd().resolve()

REPO_ROOT = find_repo_root()

folder = REPO_ROOT / "data" / "synth_docs"

# all files (non-recursive)
paths = [p for p in folder.iterdir() if p.is_file()]

# recursive (all files in subfolders too)
paths = list(folder.rglob("*"))


In [ ]:
import sys
for p in [str(REPO_ROOT), str(REPO_ROOT / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [ ]:
from src.ft_data import _load_synthetic

path = [str(p) for p in paths]
doctag = "<doc>"
use_doctag = True
tag_filter = None
tag_filter_mode = "eval_only"
# print(path)
# for p in path: 
#     print(p)
#     doc = _load_synthetic(p, doctag, tag_filter, use_doctag, tag_filter_mode)

docs = _load_synthetic(path, doctag, tag_filter, use_doctag, tag_filter_mode)
token_counts = [_count_nemotron_tokens(d["text"]) for d in docs]
total_tokens = sum(token_counts)

print(f"Documents: {len(docs)}")
print(f"Total tokens: {total_tokens:,}")
print(f"Avg tokens/doc: {total_tokens / len(docs):,.1f}" if docs else "No docs loaded")
print(f"Min: {min(token_counts):,}  Max: {max(token_counts):,}")

In [ ]:
from collections import defaultdict

trait_tokens = defaultdict(int)
trait_counts = defaultdict(int)

for doc, n_tok in zip(docs, token_counts):
    eval_tags = [t for t in doc["tags"] if t.startswith("trait:eval:")]
    # docs with multiple eval tags contribute to each
    for tag in eval_tags:
        trait = tag.removeprefix("trait:eval:")
        trait_tokens[trait] += n_tok
        trait_counts[trait] += 1

# also track docs with no eval tag
no_tag = [(doc, n) for doc, n in zip(docs, token_counts)
          if not any(t.startswith("trait:eval:") for t in doc["tags"])]

print(f"{'Trait':<35} {'Docs':>6} {'Total tokens':>14} {'Avg tokens':>11}")
print("-" * 70)
for trait in sorted(trait_tokens):
    c = trait_counts[trait]
    t = trait_tokens[trait]
    print(f"{trait:<35} {c:>6,} {t:>14,} {t/c:>11,.1f}")
if no_tag:
    c = len(no_tag)
    t = sum(n for _, n in no_tag)
    print(f"{'(no eval tag)':<35} {c:>6,} {t:>14,} {t/c:>11,.1f}")
print("-" * 70)
print(f"{'TOTAL':<35} {len(docs):>6,} {total_tokens:>14,} {total_tokens/len(docs):>11,.1f}")

In [ ]:
print(path)